# Local RAG in LangChain without PyTorch

`KjarniEmbeddings` runs the same embedding models as `HuggingFaceEmbeddings`, in your own process, without PyTorch. This notebook measures what that changes: the install, whether the vectors really are the same, and the speed. Then it builds a small retriever over Kjarni's own documentation.

Measured on Linux x86_64 (Intel i7-13700, 24 threads), Python 3.13, CPU only. Measured 2026-10-24.

## 1. What gets installed

A fresh virtual environment for each, measured as the size of `site-packages` after `pip install`:

| Install | Size | Packages |
|:--|--:|--:|
| `pip install langchain-kjarni` | 98 MB | 36 |
| `pip install langchain-huggingface sentence-transformers`, CPU-only PyTorch | 1,389 MB | 67 |
| `pip install langchain-huggingface sentence-transformers` (default) | 5,913 MB | 86 |

Without an index URL, `pip` on Linux installs the CUDA build of PyTorch, whether or not the machine has an NVIDIA GPU. [`measure_install.sh`](measure_install.sh) reproduces these numbers.

## 2. The same vectors

Both sides run all-MiniLM-L6-v2 on the same texts. If Kjarni is a drop-in replacement, the vectors should agree to floating-point precision.

In [1]:
import numpy as np
from langchain_kjarni import KjarniEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

kjarni = KjarniEmbeddings(model="minilm-l6-v2")
torch_st = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

texts = [
    "Reykjavik is the capital of Iceland.",
    "The volcano erupted for three weeks and closed the airport.",
    "Refunds are issued within 14 days of the return.",
    "How do I rerank search results with a cross-encoder?",
    "Kjarni runs embedding models without PyTorch.",
]
a = np.array(kjarni.embed_documents(texts))
b = np.array(torch_st.embed_documents(texts))
cosines = (a * b).sum(axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
print(f"vectors: {a.shape} vs {b.shape}")
print(f"largest difference in any value: {np.abs(a - b).max():.2e}")
print("cosine between the two, per text:", np.round(cosines, 7))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

vectors: (5, 384) vs (5, 384)
largest difference in any value: 1.27e-07
cosine between the two, per text: [1. 1. 1. 1. 1.]


## 3. A retriever over Kjarni's documentation

`llms-full.txt` is the whole of kjarni.ai as one Markdown file. Split into passages, indexed in LangChain's `InMemoryVectorStore` with `KjarniEmbeddings`, it answers questions about Kjarni.

In [2]:
import urllib.request
from langchain_core.vectorstores import InMemoryVectorStore

request = urllib.request.Request(
    "https://kjarni.ai/llms-full.txt", headers={"User-Agent": "langchain-kjarni-example"}
)
doc = urllib.request.urlopen(request).read().decode("utf-8")

# Paragraphs, merged up to about 400 characters per passage: short
# enough that each passage is about one thing.
passages, current = [], ""
for paragraph in (p.strip() for p in doc.split("\n\n")):
    if not paragraph:
        continue
    if current and len(current) + len(paragraph) > 400:
        passages.append(current)
        current = ""
    current = f"{current}\n\n{paragraph}" if current else paragraph
passages.append(current)
print(f"{len(doc):,} characters, {len(passages)} passages")

store = InMemoryVectorStore.from_texts(passages, embedding=kjarni)
for question in [
    "Does Kjarni need Python installed?",
    "How do I use Kjarni from a Qt Quick application?",
    "Can it run in the browser?",
]:
    best = store.similarity_search(question, k=1)[0].page_content
    print(f"\nQ: {question}\nA: {' '.join(best.split())[:300]}...")

204,774 characters, 622 passages



Q: Does Kjarni need Python installed?
A: ### Does Kjarni need Python? No. The engine is a native library, so a C#, Go, C++, Rust or command line program needs no Python installed. Kjarni also publishes Python bindings on PyPI for use from Python. ### Does Kjarni need a GPU or CUDA? No. Everything runs on CPU by default. GPU support is opti...

Q: How do I use Kjarni from a Qt Quick application?
A: The application links one shared library. On Linux keep `libkjarni_ffi.so` next to the binary with an `$ORIGIN` rpath, on Windows put `kjarni_ffi.dll` next to the `.exe`, and run `linuxdeploy`, `windeployqt` or `macdeployqt` for Qt as you normally would. Kjarni itself needs nothing else: no Python, ...

Q: Can it run in the browser?
A: Yes. The `kjarni-wasm` npm package runs embeddings, classification, reranking, search and chat in a Web Worker, on CPU with WebAssembly SIMD. The [live demo](/demo/) runs three models in the page. ### What license is Kjarni under? MIT or Apache 2.0, at your op

## 4. Speed

Embedding every passage from section 3, each side warmed up first. Then cold start: a fresh Python process that imports the integration, loads the model from the local cache and embeds one sentence.

In [3]:
import time

for name, model in [("KjarniEmbeddings", kjarni), ("HuggingFaceEmbeddings", torch_st)]:
    model.embed_documents(passages[:8])
    start = time.perf_counter()
    model.embed_documents(passages)
    elapsed = time.perf_counter() - start
    print(f"{name:22} {len(passages)} passages in {elapsed:5.2f} s  ({len(passages) / elapsed:,.0f} per second)")

KjarniEmbeddings       622 passages in 11.13 s  (56 per second)


HuggingFaceEmbeddings  622 passages in  6.22 s  (100 per second)


In [4]:
import subprocess, sys

cold = {
    "KjarniEmbeddings": "from langchain_kjarni import KjarniEmbeddings; KjarniEmbeddings().embed_query('hello')",
    "HuggingFaceEmbeddings": (
        "from langchain_huggingface import HuggingFaceEmbeddings; "
        "HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2').embed_query('hello')"
    ),
}
for name, program in cold.items():
    runs = []
    for _ in range(3):
        start = time.perf_counter()
        subprocess.run([sys.executable, "-c", program], check=True, capture_output=True)
        runs.append(time.perf_counter() - start)
    print(f"{name:22} cold start, best of 3: {min(runs):.2f} s")

KjarniEmbeddings       cold start, best of 3: 0.26 s


HuggingFaceEmbeddings  cold start, best of 3: 4.84 s


## Summary

`KjarniEmbeddings` is a drop-in replacement for `HuggingFaceEmbeddings`: the same vectors, from an install a fraction of the size, with no PyTorch, no GPU and no server, and a cold start in a fraction of a second rather than seconds. For bulk embedding of a small model like MiniLM on a many-core CPU, PyTorch's throughput is higher, as section 4 shows; where Kjarni pays off is everywhere the process starts often or the install has to stay small: scripts, command-line tools, serverless functions, containers and tests. Other models, including multilingual ones, are listed in the [Kjarni documentation](https://kjarni.ai/docs/).